In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
project_path = "/content/drive/MyDrive/Group 7 Capstone Project"
images_path = f"{project_path}/DC_images"
csv_path = f"{project_path}/scenicness_dataset.csv"

In [3]:
import pandas as pd

df = pd.read_csv(csv_path)
df.head()

,scenicness,connectivity,objects,description,id,latitude,longitude,image_path,greenery_pct,building_pct,sky_pct,road_pct,water_pct
0,0,0,"['text (Sorry, we have no imagery here)']",A blank image with a central message stating t...,5513,38.865521,-77.009026,/content/drive/MyDrive/Scenicness_Project/DC_i...,0.000977,0.00000,99.999023,0.00000,0.0
1,0,0,"[""text: 'Sorry, we have no imagery here.'""]","A blank, featureless screen with a message sta...",3274,38.939376,-77.055174,/content/drive/MyDrive/Scenicness_Project/DC_i...,0.000977,0.00000,99.999023,0.00000,0.0
2,6,2,"['trees', 'fallen branches', 'leaf litter', 'f...","A forest scene with dense trees, leaf-covered ...",8371,38.879924,-76.951403,/content/drive/MyDrive/Scenicness_Project/DC_i...,40.528320,0.00000,3.852295,0.00000,0.0
3,0,0,"[""text: 'Sorry, we have no imagery here.'""]",This image displays a blank background with a ...,3500,38.935775,-77.050558,/content/drive/MyDrive/Scenicness_Project/DC_i...,0.000977,0.00000,99.999023,0.00000,0.0
4,3,6,"['houses', 'cars', 'trash bins', 'power lines'...",A residential street lined with brick row hous...,1912,38.950170,-77.082869,/content/drive/MyDrive/Scenicness_Project/DC_i...,6.958984,17.61084,30.894531,2.39624,0.0


In [4]:
df['image_path'] = df['id'].apply(lambda x: f"DC_images/dc_{x}.jpg")

In [5]:
df.to_csv(csv_path, index=False)

In [7]:
df['image_path'].head(10)

,image_path
0,DC_images/dc_5513.jpg
1,DC_images/dc_3274.jpg
2,DC_images/dc_8371.jpg
3,DC_images/dc_3500.jpg
4,DC_images/dc_1912.jpg
5,DC_images/dc_6010.jpg
6,DC_images/dc_9401.jpg
7,DC_images/dc_5232.jpg
8,DC_images/dc_1326.jpg
9,DC_images/dc_5385.jpg


In [6]:
import os

missing = []
for p in df['image_path']:
    if not os.path.exists(f"{project_path}/{p}"):
        missing.append(p)

len(missing), missing[:10]

(0, [])

**Load CLIP + Preprocessing**

In [8]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-stxdvy_w
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-stxdvy_w
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=2b3bce7a241900b7e489c0a201a9bc5427d4acb7400106ecbc115716ebe874c9
  Stored in directory: /tmp/pip-ephem-wheel-cache-5h7hpt6f/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [9]:
# Loading CLIP
import torch
import clip
from PIL import Image
import numpy as np
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 179MiB/s]


**Embeddings Extraction**

In [10]:
import os

embeddings_dir = f"{project_path}/embeddings"
os.makedirs(embeddings_dir, exist_ok=True)

In [11]:
all_embeddings = []

for path in tqdm(df['image_path'], desc="Extracting CLIP embeddings"):
    full_path = f"{project_path}/{path}"

    try:
        image = Image.open(full_path).convert("RGB")
        image_input = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            embedding = model.encode_image(image_input)
            embedding = embedding.cpu().numpy().flatten()

        all_embeddings.append(embedding)

    except Exception as e:
        print("Error loading:", full_path, e)
        all_embeddings.append(np.zeros(512))

Extracting CLIP embeddings: 100%|██████████| 997/997 [15:06<00:00,  1.10it/s]


In [12]:
#Saving Embeddings
embeddings_array = np.vstack(all_embeddings)
np.save(f"{embeddings_dir}/clip_embeddings.npy", embeddings_array)

embeddings_array.shape

(997, 512)